## Model Comparison

The goal of this notebook is to compare the performance of the four different models in our classification task.

In [34]:
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

#### Loading the models

In [35]:
# Load the models
logisticRegression = joblib.load('../models/logistic_regression_model.joblib')
randomForest = joblib.load('../models/random_forest_model.joblib')
XGBoost = joblib.load('../models/XGBoost_model.joblib')


#### Preparing the test data

In [37]:
# Load the test data for comparison
df = pd.read_csv('../data/processed/FW_Veg_Rem_Combined_transformed.csv')
df['Vegetation'] = df['Vegetation'].astype('category')
dummy_cols = [col for col in df.columns if col.startswith('Vegetation_')]

X_onehot = df.drop(columns=['Vegetation','catastrophic','index'])
X_cat = df.drop(columns=['catastrophic','index',*dummy_cols])
y = df['catastrophic']

X_train_onehot, X_test_onehot, y_train, y_test = train_test_split(X_onehot, y, test_size=0.2, stratify=y, random_state=42)
X_train_cat, X_test_cat, y_train, y_test = train_test_split(X_cat, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train_onehot)
X_train_onehot_scaled = scaler.transform(X_train_onehot)
X_test_onehot_scaled = scaler.transform(X_test_onehot)

#### Computing the predictions

In [54]:
# Logistic Model with default threshold
y_pred_logistic = logisticRegression.predict(X_test_onehot_scaled)

# Logistic Model with custom decision threshold
y_pred_proba_logistic = logisticRegression.predict_proba(X_test_onehot_scaled)[:, 1]
logistic_threshold = 0.47
y_pred_logistic_custom = (y_pred_proba_logistic >= logistic_threshold).astype(int)

# Random Forest with default decision threshold
y_pred_random_forest = randomForest.predict(X_test_onehot)

# Random Forest with custom decision threshold
y_pred_proba_random_forest = randomForest.predict_proba(X_test_onehot)[:, 1]
rf_threshold = 0.45
y_pred_rf_custom = (y_pred_proba_random_forest >= rf_threshold).astype(int)

# XGBoost
y_pred_XGBoost = XGBoost.predict(X_test_cat)

In [57]:
# Compare accuracy, recall, precision, and f1-score
def evaluate_model(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return accuracy, recall, precision, f1

# Logistic Regression
accuracy_logistic, recall_logistic, precision_logistic, f1_logistic = evaluate_model(y_test, y_pred_logistic)
# Logistic Regression with custom threshold
accuracy_logistic_custom, recall_logistic_custom, precision_logistic_custom, f1_logistic_custom = evaluate_model(y_test, y_pred_logistic_custom)
# Random Forest
accuracy_random_forest, recall_random_forest, precision_random_forest, f1_random_forest = evaluate_model(y_test, y_pred_random_forest)
# Random Forest with custom threshold
accuracy_random_forest_custom, recall_random_forest_custom, precision_random_forest_custom, f1_random_forest_custom = evaluate_model(y_test, y_pred_rf_custom)
# XGBoost
accuracy_XGBoost, recall_XGBoost, precision_XGBoost, f1_XGBoost = evaluate_model(y_test, y_pred_XGBoost)

# Print the results in a table
results = pd.DataFrame({
    'Model': ['Logistic Regression', f'Logistic Regression with threshold:{logistic_threshold}', 'Random Forest', f'Random Forest with threshold:{rf_threshold}', 'XGBoost'],
    'Accuracy': [accuracy_logistic, accuracy_logistic_custom, accuracy_random_forest, accuracy_random_forest_custom, accuracy_XGBoost],
    'Recall': [recall_logistic, recall_logistic_custom, recall_random_forest, recall_random_forest_custom, recall_XGBoost],
    'Precision': [precision_logistic, precision_logistic_custom, precision_random_forest, precision_random_forest_custom, precision_XGBoost],
    'F1-Score': [f1_logistic, f1_logistic_custom, f1_random_forest, f1_random_forest_custom, f1_XGBoost]
})
results = results.set_index('Model')
results = results.sort_values(by='F1-Score', ascending=False)
display(results)


,Accuracy,Recall,Precision,F1-Score
Model,,,,
XGBoost,0.684493,0.696934,0.534047,0.604714
Random Forest with threshold:0.45,0.681134,0.692666,0.530303,0.600707
Random Forest,0.732061,0.536671,0.633532,0.581092
Logistic Regression with threshold:0.47,0.602392,0.647652,0.448656,0.530094
Logistic Regression,0.658022,0.542103,0.505793,0.523319


np.int64(0)